# 1. Data stage (CPU session, no GPU quota used)

Kaggle settings (right panel): **Accelerator: None**, **Internet: On**.
Add your Hugging Face token under **Add-ons -> Secrets** with the name `HF_TOKEN`
(only English/Svarah needs it).

Run with **Save Version -> Save & Run All (Commit)**. Every stage is a `%%bash` cell with
`set -euo pipefail`, and each is followed by a Python check, so any failure stops the run
(status ERROR) instead of silently continuing. Output streams live to the Logs tab.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/snehithareddy28/fastspeech2-assignment.git"
os.chdir("/kaggle/working")
if not os.path.exists("fastspeech2-assignment"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/kaggle/working/fastspeech2-assignment")
subprocess.run(["git", "pull"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
os.environ["TQDM_MININTERVAL"] = "60"   # keep progress bars from flooding the log
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret (needed for English):", e)

In [ ]:
%%bash
set -euo pipefail
pip install -q -r requirements.txt
bash scripts/setup_mfa.sh

In [ ]:
%%bash
set -euo pipefail
bash scripts/run_pipeline.sh data telugu

In [ ]:
import json
s = json.load(open("data/Telugu/preprocessed/summary.json"))
print(json.dumps(s, indent=1)); print(open("Results/Telugu/alignment_coverage.txt").read())
assert s["train_items"] > 1000, "too few Telugu training items"

In [ ]:
%%bash
set -euo pipefail
bash scripts/run_pipeline.sh data english

In [ ]:
s = json.load(open("data/English/preprocessed/summary.json"))
print(json.dumps(s, indent=1)); print(open("Results/English/alignment_coverage.txt").read())
for L in ["English", "Telugu"]:
    d = json.load(open(f"Results/{L}/data_selection_summary.json"))
    print(L, {k: d.get(k) for k in ["speaker_id_method", "speakers_chosen", "speaker_used",
                                    "train_hours", "val_hours", "train_utts", "val_utts"]})
assert s["train_items"] > 1000, "too few English training items"

In [ ]:
%%bash
set -euo pipefail
# both languages have features: drop what training does not need (MFA env, raw audio)
rm -rf .mfa_env .micromamba mfa_models data/*/raw/*/ data/*/TextGrid
du -sh data/*/preprocessed